# DiLoCo Distributed Low-Communication Training

**Audience:** ML practitioners who know Python and basic linear algebra.  
**Prerequisites:** NumPy, optimization basics, and the project README.  
**Learning goals:** implement the core algorithm, validate invariants, and evaluate a deterministic end-to-end run.

## Outline
1. Configure the reference system.
2. Execute its core training or simulation loop.
3. Verify numerical and behavioral assertions.
4. Try the exercise and inspect pitfalls.


In [1]:
from pathlib import Path
import sys, numpy as np
ROOT=Path.cwd()
if not (ROOT/'src').exists():
    ROOT=Path('/Users/shivamkumar/Documents/Codex/ml/implementations/diloco-distributed-low-communication-training-of-language-models')
sys.path.insert(0,str(ROOT/'src'))
np.set_printoptions(precision=3,suppress=True)


## 1. Build heterogeneous worker shards
Each worker sees a shifted distribution, while all optimize one linear model.


In [2]:
from diloco.core import *
cfg=DiLoCoConfig(rounds=30)
shards, truth=make_shards(cfg)
print('worker means:', [round(float(x.mean()),3) for x,_ in shards], 'truth:', truth)


worker means: [-0.39, -0.144, -0.12, 0.296] truth: [0.333 0.667 1.   ]


## 2. Run inner and outer optimization
Workers take many local steps; the coordinator treats parameter deltas as pseudo-gradients.


In [3]:
result=train(cfg)
print('initial/final loss:', round(result['history'][0]['loss'],5), round(result['history'][-1]['loss'],5))
print('learned:', result['weights'].round(3), 'communication reduction:', result['local_steps']/result['communication_rounds'])


initial/final loss: 0.39534 0.01072
learned: [0.306 0.744 1.067] communication reduction: 8.0


In [4]:
assert result['history'][-1]['loss'] < result['history'][0]['loss']*.2
assert np.linalg.norm(result['weights']-truth)<.2
print('evaluation passed')


evaluation passed


## Exercise
Change one capacity, rank, learning-rate, sample-size, or risk parameter in `config.json`. Predict the direction of the primary metric before rerunning.


In [5]:
# Answer scaffold: record the parameter, prediction, and observed metric.
exercise = {'parameter': 'choose one', 'prediction': 'state a direction', 'observed': None}
print(exercise)


{'parameter': 'choose one', 'prediction': 'state a direction', 'observed': None}


## Pitfalls and extensions
**Pitfall:** comparing runs with different seeds or compute budgets can masquerade as an algorithmic gain. Keep the seed and budget fixed.  
**Extension:** add a larger held-out scenario and report both quality and resource cost.
